# 🅿️ ParkVision AI: Intelligent Urban Parking Analytics & Space Optimisation
### Deep Learning Model Training Pipeline (Google Colab Edition)
**Company:** UrbanFlow AI Pvt. Ltd. · Smart Mobility & Urban Analytics  
**Models Trained:** **YOLOv8** (State-of-the-Art Vision Backbone) & **MobileNetV2** (Lightweight Transfer Learning)  
**Dataset:** PKLot Multi-Weather Benchmark (Sunny, Cloudy, Rainy) with 70% Train / 15% Val / 15% Test  
**Safety Guarantee:** Built-in Anti-Data Loss Protection (Google Drive Auto-Mount + Instant Direct Download)  
**Resumption Guarantee:** If training stops unexpectedly, re-running the cell automatically **resumes from the last completed epoch** without restarting from scratch!

---

### Instructions for Running in Google Colab:
1. **Enable GPU Acceleration**: Navigate to **Runtime** → **Change runtime type** → Select **T4 GPU** → Click **Save**.
2. **Execute Sequentially**: Run each cell from top to bottom (`Shift + Enter`).
3. **Google Drive Sync**: Cell 2 prompts you to mount Google Drive. This guarantees that all model weights (`best.pt`, `parkvision_mobilenetv2.keras`) and checkpoints (`last.pt`) are mirrored directly to your Google Drive and will **never vanish** even if your session disconnects.
4. **Final Export**: The last cell automatically packages the trained models, evaluation plots, and metrics into a `.zip` archive and triggers an immediate browser download.

In [ ]:
# 1. System Environment & Hardware Acceleration Check
import os
import sys
import tensorflow as tf

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU Detected: {len(gpus)} device(s) available.")
    for gpu in gpus:
        print(f"   Device: {gpu.name}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("   Memory growth enabled successfully.")
    except Exception as e:
        print(f"   Note on memory growth: {e}")
else:
    print("\n⚠️ Running on CPU. (Tip: Change runtime type to T4 GPU for 10x faster training).")

# Install ultralytics for YOLO training
!pip install -q ultralytics
print("✅ Ultralytics YOLO installed and ready.")

## 🛡️ Anti-Data Loss Layer: Persistent Cloud Storage
Mount Google Drive to guarantee that hours of training never result in vanished data. All checkpoints will be mirrored directly to your Drive in real-time.

In [ ]:
# 2. Configure Persistent Cloud Storage (Google Drive)
from pathlib import Path
import shutil

USE_DRIVE = False
DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")

try:
    from google.colab import drive
    print("Mounting Google Drive to ensure trained models are never lost...")
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = True
    print(f"✅ Google Drive mounted successfully! Checkpoints will persist in:\n   {DRIVE_BASE}")
except Exception as e:
    print(f"ℹ️ Google Drive mount skipped or not running in Colab ({e}). Checkpoints saved locally.")
    DRIVE_BASE = Path("./colab_checkpoints")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Working directories
WORK_DIR = Path("./parkvision_run")
DATASET_DIR = WORK_DIR / "dataset"
MODELS_DIR = WORK_DIR / "models"
REPORTS_DIR = WORK_DIR / "reports"

for d in [DATASET_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"Working directories initialized under {WORK_DIR.resolve()}")

## 📥 Data Collection: PKLot Dataset & Balanced Multi-Weather Sampling
The assignment requires a real-world parking lot dataset captured across varied environmental conditions (**sunny, cloudy, rainy**) with at least 100 images per class (occupied and empty).

The code below checks for existing PKLot files or generates a calibrated multi-weather parking bay dataset matching PKLot specifications so you can execute end-to-end without interruption.

In [ ]:
# 3. Dataset Acquisition: Real-world PKLot Dataset
import hashlib
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

RAW_DIR = WORK_DIR / "raw_dataset"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def ensure_dataset(raw_dir: Path, target_per_class=300):
    """Ensure we have a diverse, balanced multi-weather dataset with empty & occupied slots.
    Automatically detects user dataset archives uploaded to Google Drive or Colab.
    """
    import zipfile
    
    # Check if user uploaded a zip archive or folder to Google Drive or /content
    search_locations = []
    if USE_DRIVE:
        search_locations.extend([DRIVE_BASE, DRIVE_BASE.parent, Path("/content/drive/MyDrive")])
    search_locations.append(Path("/content"))
    
    for loc in search_locations:
        if not loc.exists():
            continue
        # Check for zip archives (e.g. PKLot.zip, dataset.zip, raw_dataset.zip)
        for z in loc.glob("*.zip"):
            if "submission" in z.name.lower() or "bundle" in z.name.lower():
                continue
            # Check if this zip has images
            try:
                with zipfile.ZipFile(z, "r") as zip_ref:
                    namelist = zip_ref.namelist()
                    if any("empty" in n.lower() or "occupied" in n.lower() for n in namelist):
                        print(f"📦 Found dataset archive in {loc.name}: {z.name}. Extracting...")
                        zip_ref.extractall(raw_dir)
                        break
            except Exception as e:
                pass
        
        # Check for uncompressed folders
        drive_raw = loc / "raw_dataset"
        if drive_raw.exists() and drive_raw.is_dir() and not (raw_dir / "empty").exists():
            print(f"📂 Found raw_dataset folder in {loc.name}! Copying images...")
            shutil.copytree(drive_raw, raw_dir, dirs_exist_ok=True)

    existing_empty = list(raw_dir.rglob("empty/*.jpg")) + list(raw_dir.rglob("Empty/*.jpg")) + list(raw_dir.rglob("empty/*.png"))
    existing_occupied = list(raw_dir.rglob("occupied/*.jpg")) + list(raw_dir.rglob("Occupied/*.jpg")) + list(raw_dir.rglob("occupied/*.png"))
    
    if len(existing_empty) >= 100 and len(existing_occupied) >= 100:
        print(f"✅ Real PKLot dataset verified: {len(existing_empty)} empty, {len(existing_occupied)} occupied.")
        return

    raise FileNotFoundError(
        "❌ Real PKLot dataset not found in /content or Google Drive! "
        "Please ensure 'data.zip' or 'data_extended_10k.zip' is uploaded to Colab or your Google Drive. "
        "Strictly real camera photos are required; synthetic generation is disabled."
    )

ensure_dataset(RAW_DIR, target_per_class=300)

## ⚙️ Data Preprocessing & 70% / 15% / 15% Split
Per the assignment requirements:
- Balance classes (at least 100 images per class).
- Uniformly resize all images to **224×224 pixels**.
- Partition into **70% Training**, **15% Validation**, and **15% Testing**.
- Organize into standard `/empty` and `/occupied` directory hierarchies.

In [ ]:
# 4. Partition and Resize Dataset into 70/15/15 Train/Val/Test
import json
import shutil
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass
from PIL import Image

# Self-healing environment check
WORK_DIR = Path("./parkvision_run")
DATASET_DIR = WORK_DIR / "dataset"
MODELS_DIR = WORK_DIR / "models"
REPORTS_DIR = WORK_DIR / "reports"
RAW_DIR = WORK_DIR / "raw_dataset"
for d in [WORK_DIR, DATASET_DIR, MODELS_DIR, REPORTS_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")
USE_DRIVE = Path("/content/drive/MyDrive").exists()

@dataclass
class SlotItem:
    path: Path
    label: str
    weather: str
    digest: str = ""

def collect_images(source: Path):
    items = {"empty": [], "occupied": []}
    for p in source.rglob("*"):
        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"} or not p.is_file():
            continue
        p_str = str(p).lower()
        label = "empty" if "empty" in p_str else ("occupied" if "occupied" in p_str else None)
        if label is None:
            continue
        weather = "sunny" if "sunny" in p_str else ("cloudy" if "cloudy" in p_str else ("rainy" if "rainy" in p_str else "general"))
        items[label].append(SlotItem(p, label, weather))
    return items

def build_partitions(source: Path, out_dir: Path, target_per_class=300, ratios=(0.70, 0.15, 0.15)):
    # 1. Check if user provided pre-partitioned 70/15/15 dataset (e.g. from data.zip)
    candidates = [
        WORK_DIR / "data" / "processed",
        source / "data" / "processed",
        source / "processed",
        Path("/content/data/processed")
    ]
    if USE_DRIVE:
        candidates.append(DRIVE_BASE / "data" / "processed")

    for cand in candidates:
        if cand.exists() and (cand / "train" / "empty").exists() and (cand / "val" / "empty").exists():
            print(f"✅ Found pre-partitioned 70/15/15 dataset in {cand}! Syncing directly...")
            if out_dir.exists():
                shutil.rmtree(out_dir)
            shutil.copytree(cand, out_dir)
            summary = {s: {l: len(list((out_dir / s / l).glob('*.jpg'))) for l in ['empty', 'occupied']} for s in ['train', 'val', 'test']}
            print("✅ Partition summary (using pre-partitioned real PKLot dataset):")
            print(json.dumps(summary, indent=2))
            return summary

    if out_dir.exists():
        shutil.rmtree(out_dir)
        
    items = collect_images(source)
    counts = {k: len(v) for k, v in items.items()}
    print(f"Collected raw items: {counts}")
    assert min(counts.values()) >= 100, f"Error: Need at least 100 images per class (found {counts})"
    
    n_use = min(target_per_class, min(counts.values()))
    rng = random.Random(42)
    splits = {"train": defaultdict(list), "val": defaultdict(list), "test": defaultdict(list)}
    
    for label, list_items in items.items():
        rng.shuffle(list_items)
        seen = set()
        chosen = []
        for it in list_items:
            h = hashlib.sha256(it.path.read_bytes()).hexdigest()
            if h in seen:
                continue
            seen.add(h)
            it.digest = h
            chosen.append(it)
            if len(chosen) >= n_use:
                break
                
        n_tr = round(ratios[0] * len(chosen))
        n_va = round(ratios[1] * len(chosen))
        splits["train"][label] = chosen[:n_tr]
        splits["val"][label] = chosen[n_tr:n_tr + n_va]
        splits["test"][label] = chosen[n_tr + n_va:]
        
    for split_name, class_dict in splits.items():
        for label, img_list in class_dict.items():
            dest_dir = out_dir / split_name / label
            dest_dir.mkdir(parents=True, exist_ok=True)
            for it in img_list:
                with Image.open(it.path) as img:
                    img.convert("RGB").resize((224, 224), Image.BILINEAR).save(
                        dest_dir / f"{label}_{it.digest[:12]}.jpg", quality=95)
                        
    summary = {s: {l: len(list((out_dir / s / l).glob('*.jpg'))) for l in ['empty', 'occupied']} for s in ['train', 'val', 'test']}
    print("✅ Partition summary:")
    print(json.dumps(summary, indent=2))
    return summary

build_partitions(RAW_DIR, DATASET_DIR, target_per_class=300)

## 🚀 SOTA Model: YOLOv8 Training with Automatic Resumption
Train the **highest-accuracy YOLOv8 Nano Classifier** directly on the 70/15/15 dataset.
- **Anti-Interruption Guarantee:** Automatically checks if `last.pt` exists in your Google Drive or local storage. If training was interrupted, it automatically activates `resume=True` and continues from the exact epoch where it stopped!
- **Real-Time Drive Mirroring:** Saves `best.pt` and `last.pt` directly to your mounted Google Drive after every single epoch.
- **Unseen Test Evaluation:** Evaluates performance on the 15% held-out test split.

In [ ]:
# 5. YOLOv8 Training with Auto-Resumption & Drive Sync
import os
import shutil
import zipfile
from pathlib import Path
from ultralytics import YOLO

# Self-healing environment check & path resolution
WORK_DIR = Path("./parkvision_run")
MODELS_DIR = WORK_DIR / "models"
DATASET_DIR = WORK_DIR / "dataset"
REPORTS_DIR = WORK_DIR / "reports"
for d in [WORK_DIR, MODELS_DIR, DATASET_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")
USE_DRIVE = False
if Path("/content/drive/MyDrive").exists():
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = True

# Auto-detect data.zip if dataset is not yet unpacked
if not (DATASET_DIR / "train" / "empty").exists():
    print("ℹ️ Auto-detecting and setting up dataset from data.zip or Google Drive...")
    zip_candidates = [
        Path("/content/data.zip"),
        Path("./data.zip"),
        DRIVE_BASE / "data.zip" if USE_DRIVE else None,
        Path("/content/drive/MyDrive/data.zip") if USE_DRIVE else None
    ]
    for dz in [c for c in zip_candidates if c and c.exists()]:
        print(f"📦 Extracting {dz}...")
        try:
            with zipfile.ZipFile(dz, "r") as zf:
                zf.extractall("/content")
            break
        except Exception as e:
            print(f"Extraction note: {e}")
    
    # Check if data/processed is available
    for cand in [Path("/content/data/processed"), Path("./data/processed"), WORK_DIR / "data" / "processed"]:
        if cand.exists() and (cand / "train" / "empty").exists():
            if DATASET_DIR.exists():
                shutil.rmtree(DATASET_DIR)
            shutil.copytree(cand, DATASET_DIR)
            print(f"✅ Loaded 70/15/15 PKLot dataset into {DATASET_DIR}!")
            break

YOLO_BEST = MODELS_DIR / "best.pt"
YOLO_LAST = MODELS_DIR / "last.pt"
DRIVE_YOLO_BEST = DRIVE_BASE / "best.pt"
DRIVE_YOLO_LAST = DRIVE_BASE / "last.pt"

# Check for existing checkpoint to resume
resume_ckpt = None
if DRIVE_YOLO_LAST.exists():
    resume_ckpt = DRIVE_YOLO_LAST
    print(f"🔄 Found checkpoint in Google Drive! Resuming from: {resume_ckpt}")
elif YOLO_LAST.exists():
    resume_ckpt = YOLO_LAST
    print(f"🔄 Found local checkpoint! Resuming from: {resume_ckpt}")

if resume_ckpt:
    yolo_model = YOLO(str(resume_ckpt))
    results = yolo_model.train(
        data=str(DATASET_DIR),
        epochs=25,
        imgsz=224,
        batch=32,
        resume=True
    )
else:
    print("🚀 Starting fresh YOLOv8 training (25 Epochs)...")
    yolo_model = YOLO("yolov8n-cls.pt")
    results = yolo_model.train(
        data=str(DATASET_DIR),
        epochs=25,
        imgsz=224,
        batch=32,
        project=str(WORK_DIR / "runs"),
        name="parkvision_yolo",
        exist_ok=True,
        save=True
    )

# Save best and last weights locally and mirror to Drive
run_best = getattr(yolo_model.trainer, "best", None)
if run_best and Path(run_best).exists():
    shutil.copy2(run_best, YOLO_BEST)
    if USE_DRIVE:
        shutil.copy2(run_best, DRIVE_YOLO_BEST)
        print(f"🛡️ Mirrored YOLO best model to Google Drive: {DRIVE_YOLO_BEST}")

run_last = getattr(yolo_model.trainer, "last", None)
if run_last and Path(run_last).exists():
    shutil.copy2(run_last, YOLO_LAST)
    if USE_DRIVE:
        shutil.copy2(run_last, DRIVE_YOLO_LAST)
        print(f"🛡️ Mirrored YOLO checkpoint to Google Drive: {DRIVE_YOLO_LAST}")

print(f"✅ YOLOv8 Training Complete! Best weights: {YOLO_BEST}")

## 📊 YOLOv8 Test Set Evaluation (15% Unseen Data)
Evaluate the trained YOLO model against the held-out test dataset.

In [ ]:
# 6. Evaluate YOLO on the 15% Unseen Test Set
import json
import shutil
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from datetime import datetime
from ultralytics import YOLO

# Self-healing environment check
WORK_DIR = Path("./parkvision_run")
MODELS_DIR = WORK_DIR / "models"
DATASET_DIR = WORK_DIR / "dataset"
REPORTS_DIR = WORK_DIR / "reports"
for d in [WORK_DIR, MODELS_DIR, DATASET_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")
USE_DRIVE = Path("/content/drive/MyDrive").exists()

YOLO_BEST = MODELS_DIR / "best.pt"
if not YOLO_BEST.exists() and USE_DRIVE and (DRIVE_BASE / "best.pt").exists():
    shutil.copy2(DRIVE_BASE / "best.pt", YOLO_BEST)

if not (DATASET_DIR / "test").exists() and Path("/content/data/processed/test").exists():
    DATASET_DIR = Path("/content/data/processed")
from datetime import datetime

test_empty = list((DATASET_DIR / "test" / "empty").glob("*.jpg"))
test_occupied = list((DATASET_DIR / "test" / "occupied").glob("*.jpg"))
test_items = [(f, 0) for f in test_empty] + [(f, 1) for f in test_occupied]

eval_yolo = YOLO(str(YOLO_BEST))
names = getattr(eval_yolo, "names", {0: "empty", 1: "occupied"})
occ_idx = 1
for k, v in names.items():
    if str(v).lower() == "occupied":
        occ_idx = int(k)
        break

y_true, y_pred, y_probs = [], [], []
for img_path, label in test_items:
    r = eval_yolo.predict(str(img_path), imgsz=224, verbose=False)[0]
    prob = float(r.probs.data[occ_idx].cpu().numpy())
    pred = 1 if prob >= 0.50 else 0
    y_probs.append(prob)
    y_pred.append(pred)
    y_true.append(label)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cr_text = classification_report(y_true, y_pred, target_names=["empty", "occupied"], digits=4, zero_division=0)

print("=" * 60)
print("             YOLOv8 TEST SET EVALUATION REPORT")
print("=" * 60)
print(cr_text)
print(f"Test Accuracy: {acc:.4%}")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["empty", "occupied"], yticklabels=["empty", "occupied"], ax=ax)
ax.set_title("YOLOv8 Test Set Confusion Matrix", fontweight="bold")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()

YOLO_CM_PATH = REPORTS_DIR / "confusion_matrix.png"
fig.savefig(YOLO_CM_PATH, dpi=150)
fig.savefig(REPORTS_DIR / "yolo_confusion_matrix.png", dpi=150)
if USE_DRIVE:
    shutil.copy2(YOLO_CM_PATH, DRIVE_BASE / "confusion_matrix.png")
    shutil.copy2(YOLO_CM_PATH, DRIVE_BASE / "yolo_confusion_matrix.png")
plt.show()

# Save classification report
(REPORTS_DIR / "classification_report.txt").write_text(cr_text)
if USE_DRIVE:
    shutil.copy2(REPORTS_DIR / "classification_report.txt", DRIVE_BASE / "classification_report.txt")

metrics = {
    "model": "YOLOv8n-cls",
    "model_architecture": "YOLOv8 Nano Classifier",
    "accuracy": float(acc),
    "precision_occupied": float(prec),
    "recall_occupied": float(rec),
    "f1_occupied": float(f1),
    "confusion_matrix": cm.tolist(),
    "test_images": len(y_true),
    "evaluated_at": datetime.now().isoformat(timespec="seconds")
}
(REPORTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(REPORTS_DIR / "yolo_metrics.json").write_text(json.dumps(metrics, indent=2))
if USE_DRIVE:
    shutil.copy2(REPORTS_DIR / "metrics.json", DRIVE_BASE / "metrics.json")
    shutil.copy2(REPORTS_DIR / "metrics.json", DRIVE_BASE / "yolo_metrics.json")
print("✅ Saved YOLO metrics and confusion matrix to both local and Google Drive.")

## 🔄 Continuous Progressive Training (Multi-GB Extended Dataset)
Use this cell to scale up training on the **large 3.9 GB dataset** (or any extended chunks) continuously!
- **Zero Redundancy:** Automatically starts from `best.pt` (accumulating all prior learning from the initial 1,000 images).
- **Batch-by-Batch Improvisation:** As each chunk of data (e.g. 10,000 images from `data_extended*.zip`) finishes training, the model's weights improve and are **immediately mirrored to Google Drive**.
- **Safe Interruption:** You can stop this cell at ANY time (or let it run until your Colab session ends). Even if you train on 1 GB of data and stop, the improved model weights are preserved and ready for deployment!

In [ ]:
# 7. Progressive Continual Training Loop
import os
import shutil
import zipfile
from pathlib import Path
from ultralytics import YOLO

# Self-healing environment check
WORK_DIR = Path("./parkvision_run")
MODELS_DIR = WORK_DIR / "models"
REPORTS_DIR = WORK_DIR / "reports"
for d in [WORK_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")
USE_DRIVE = Path("/content/drive/MyDrive").exists()

YOLO_BEST = MODELS_DIR / "best.pt"
if not YOLO_BEST.exists() and USE_DRIVE and (DRIVE_BASE / "best.pt").exists():
    shutil.copy2(DRIVE_BASE / "best.pt", YOLO_BEST)

# 1. Search for extended chunk zips in Colab and Google Drive
search_dirs = [Path("/content"), Path("."), DRIVE_BASE] if USE_DRIVE else [Path("/content"), Path(".")]
chunk_zips = []
for sd in search_dirs:
    if sd.exists():
        for z in sd.glob("data_extended*.zip"):
            if z not in chunk_zips:
                chunk_zips.append(z)

print(f"📦 Found {len(chunk_zips)} extended chunk archive(s) for progressive training:")
for z in chunk_zips:
    print(f"   - {z.name} ({z.stat().st_size / (1024*1024):.2f} MB)")

if not chunk_zips:
    print("ℹ️ No 'data_extended*.zip' found in /content or Google Drive.")
    print("   Upload 'data_extended_10k.zip' (or other chunks) to Colab or Drive to train further.")
else:
    print(f"\n🚀 Starting Progressive Training from base weights: {YOLO_BEST}...")
    temp_progressive = WORK_DIR / "progressive_temp"
    
    for idx, czip in enumerate(chunk_zips):
        print(f"\n" + "=" * 65)
        print(f"▶️ PROCESSING CHUNK {idx + 1}/{len(chunk_zips)}: {czip.name}")
        print("=" * 65)
        
        if temp_progressive.exists():
            shutil.rmtree(temp_progressive)
        temp_progressive.mkdir(parents=True, exist_ok=True)
        
        # Unpack chunk
        print(f"Extracting {czip.name}...")
        with zipfile.ZipFile(czip, "r") as zf:
            zf.extractall(temp_progressive)
            
        chunk_dataset = temp_progressive / "dataset" if (temp_progressive / "dataset").exists() else temp_progressive
        
        # Train on chunk using current best weights as backbone
        prog_model = YOLO(str(YOLO_BEST))
        prog_results = prog_model.train(
            data=str(chunk_dataset),
            epochs=3,
            imgsz=224,
            batch=32,
            lr0=0.0005,
            project=str(WORK_DIR / "runs"),
            name=f"prog_chunk_{idx+1}",
            exist_ok=True,
            save=True
        )
        
        # Update weights and immediately mirror to Google Drive
        new_best = getattr(prog_model.trainer, "best", None)
        if new_best and Path(new_best).exists():
            shutil.copy2(new_best, YOLO_BEST)
            print(f"✅ Chunk {idx + 1} complete! Updated best model: {YOLO_BEST}")
            if USE_DRIVE:
                shutil.copy2(new_best, DRIVE_BASE / "best.pt")
                print(f"🛡️ Mirrored improved model to Google Drive: {DRIVE_BASE / 'best.pt'}")
                
        # Clean up chunk to save disk space
        shutil.rmtree(temp_progressive, ignore_errors=True)
        
    print("\n🎉 All available extended chunks trained successfully!")

## 🧠 Secondary Model: MobileNetV2 Transfer Learning
We also train a MobileNetV2 classifier with data augmentation (flip, rotation, brightness, contrast) for comparison.

In [ ]:
# 7. MobileNetV2 Pipeline with In-Memory Caching
import shutil
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers

# Self-healing environment check
WORK_DIR = Path("./parkvision_run")
MODELS_DIR = WORK_DIR / "models"
DATASET_DIR = WORK_DIR / "dataset"
REPORTS_DIR = WORK_DIR / "reports"
for d in [WORK_DIR, MODELS_DIR, DATASET_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")
USE_DRIVE = Path("/content/drive/MyDrive").exists()

if not (DATASET_DIR / "train").exists() and Path("/content/data/processed/train").exists():
    DATASET_DIR = Path("/content/data/processed")

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
CLASS_NAMES = ["empty", "occupied"]

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "train", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", class_names=CLASS_NAMES, shuffle=True, seed=42)
raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "val", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", class_names=CLASS_NAMES, shuffle=False, seed=42)
raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", class_names=CLASS_NAMES, shuffle=False, seed=42)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = raw_val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = raw_test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Build Model with Augmentation
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.1),
], name="augmentation_pipeline")

base_mobilenet = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_mobilenet.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
x = base_mobilenet(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

mb_model = tf.keras.Model(inputs, outputs, name="ParkVision_MobileNetV2")
mb_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
)

print("Starting MobileNetV2 Head Training (15 Epochs)...")
history1 = mb_model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

# Phase 2 Fine-Tuning
base_mobilenet.trainable = True
for layer in base_mobilenet.layers[:100]:
    layer.trainable = False
for layer in base_mobilenet.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

mb_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy"])
print("Starting MobileNetV2 Fine-Tuning (10 Epochs)...")
history2 = mb_model.fit(train_ds, validation_data=val_ds, epochs=10)

MB_FINAL_PATH = MODELS_DIR / "parkvision_mobilenetv2.keras"
mb_model.save(MB_FINAL_PATH)
if USE_DRIVE:
    shutil.copy2(MB_FINAL_PATH, DRIVE_BASE / "parkvision_mobilenetv2.keras")
    print(f"🛡️ MobileNetV2 saved to Google Drive: {DRIVE_BASE / 'parkvision_mobilenetv2.keras'}")

## 🚦 Integration & Parking Insight Logic (Step 4 Verification)
Transforms model predictions into actionable urban parking insights:
- **Total Slots, Occupied Slots, Available Slots**
- **Occupancy Percentage**
- **Congestion Classification**: Low (<40%), Moderate (40–75%), High (>75%)
- **Action Recommendations**: "Slots available - proceed" or "Parking full - try another area"

In [ ]:
# 8. Parking Intelligence & Decision Recommendations (Step 4 Logic)
def calculate_parking_insights(statuses):
    total = len(statuses)
    occupied = sum(1 for s in statuses if s == "occupied")
    available = total - occupied
    occupancy_pct = round(100.0 * occupied / total, 1) if total > 0 else 0.0
    
    if occupancy_pct < 40.0:
        congestion = "Low"
    elif occupancy_pct <= 75.0:
        congestion = "Moderate"
    else:
        congestion = "High"
        
    if total == 0:
        rec, status_type = "No parking slots detected.", "info"
    elif available == 0:
        rec, status_type = "Parking full - try another area.", "error"
    elif congestion == "High":
        rec, status_type = f"Only {available} slot(s) left - parking is almost full, consider another area.", "warning"
    else:
        rec, status_type = f"Slots available ({available} free) - proceed.", "success"
        
    return {
        "total": total,
        "occupied": occupied,
        "available": available,
        "occupancy_pct": occupancy_pct,
        "congestion_level": congestion,
        "recommendation": rec,
        "status_type": status_type
    }

scenarios = {
    "Morning Arrival (Low)": ["empty"] * 18 + ["occupied"] * 6,
    "Midday Peak (Moderate)": ["occupied"] * 15 + ["empty"] * 9,
    "Evening Full Capacity (High)": ["occupied"] * 23 + ["empty"] * 1
}

print(f"{'Scenario':<30} | {'Total':<6} | {'Occ':<5} | {'Free':<5} | {'Pct':<6} | {'Level':<10} | {'Recommendation'}")
print("-" * 105)
for name, sample_statuses in scenarios.items():
    res = calculate_parking_insights(sample_statuses)
    print(f"{name:<30} | {res['total']:<6} | {res['occupied']:<5} | {res['available']:<5} | {res['occupancy_pct']:<5}% | {res['congestion_level']:<10} | {res['recommendation']}")

## 📦 Triple-Layer Export: Instant Direct Download
Packages both the trained **YOLOv8** model (`best.pt`) and **MobileNetV2** model (`parkvision_mobilenetv2.keras`), confusion matrix, training curve, and metrics into `ParkVision_AI_Submission_Bundle.zip` and triggers an immediate browser download.

In [ ]:
# 9. Package and Download Trained Models & Submission Bundle
import shutil
import zipfile
from pathlib import Path

# Self-healing environment check
WORK_DIR = Path("./parkvision_run")
MODELS_DIR = WORK_DIR / "models"
REPORTS_DIR = WORK_DIR / "reports"
for d in [WORK_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = Path("/content/drive/MyDrive/ParkVision_AI_Model_Checkpoints")
USE_DRIVE = Path("/content/drive/MyDrive").exists()

YOLO_BEST = MODELS_DIR / "best.pt"
YOLO_LAST = MODELS_DIR / "last.pt"
MB_FINAL_PATH = MODELS_DIR / "parkvision_mobilenetv2.keras"

# Mirror from Drive if not locally present
if not YOLO_BEST.exists() and USE_DRIVE and (DRIVE_BASE / "best.pt").exists():
    shutil.copy2(DRIVE_BASE / "best.pt", YOLO_BEST)
if not YOLO_LAST.exists() and USE_DRIVE and (DRIVE_BASE / "last.pt").exists():
    shutil.copy2(DRIVE_BASE / "last.pt", YOLO_LAST)
if not MB_FINAL_PATH.exists() and USE_DRIVE and (DRIVE_BASE / "parkvision_mobilenetv2.keras").exists():
    shutil.copy2(DRIVE_BASE / "parkvision_mobilenetv2.keras", MB_FINAL_PATH)

BUNDLE_NAME = "ParkVision_AI_Submission_Bundle.zip"
bundle_path = WORK_DIR / BUNDLE_NAME

print("Creating submission bundle archive...")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Add YOLO model
    if YOLO_BEST.exists():
        zipf.write(YOLO_BEST, arcname="models/best.pt")
        print("  + models/best.pt (YOLOv8)")
    if YOLO_LAST.exists():
        zipf.write(YOLO_LAST, arcname="models/last.pt")
        print("  + models/last.pt (YOLOv8 checkpoint)")
        
    # Add MobileNet model
    if MB_FINAL_PATH.exists():
        zipf.write(MB_FINAL_PATH, arcname="models/parkvision_mobilenetv2.keras")
        print("  + models/parkvision_mobilenetv2.keras (MobileNetV2)")
        
    # Add reports and visualizations
    for report_file in REPORTS_DIR.glob("*.*"):
        zipf.write(report_file, arcname=f"reports/{report_file.name}")
        print(f"  + reports/{report_file.name}")

if USE_DRIVE:
    shutil.copy2(bundle_path, DRIVE_BASE / BUNDLE_NAME)
    print(f"\n🛡️ Mirror copy saved to Google Drive: {DRIVE_BASE / BUNDLE_NAME}")

print(f"\n✅ Bundle successfully created ({bundle_path.stat().st_size / (1024*1024):.2f} MB).")
print("Triggering direct browser download now...")

try:
    from google.colab import files
    files.download(str(bundle_path))
    print("🚀 Browser download dialog opened! Keep the downloaded zip safe for your submission.")
except Exception as e:
    print(f"Note on direct download: {e}. You can download {bundle_path} manually from the file explorer.")